# step 1

In [1]:
import cv2
import numpy as np

# Step 1: Edge extraction and noise removal using the Canny algorithm
def preprocess_image(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    edges = cv2.Canny(blurred, 120, 150)
    return edges

# Open the laptop's camera
cap = cv2.VideoCapture(0)

while True:
    # Capture frame-by-frame
    ret, frame = cap.read()

    # Preprocess the frame
    edges = preprocess_image(frame)

    # Display the original and edges frames
    cv2.imshow('Original Frame', frame)
    cv2.imshow('Edges', edges)

    # Check if 's' is pressed
    key = cv2.waitKey(1) & 0xFF
    if key == ord('s'):
        # Save the original frame
        cv2.imwrite('original_frame.png', frame)
        
        # Save the edge frame
        cv2.imwrite('edge_frame.png', edges)
        
    elif key == ord('q'):
        # Break the loop if 'q' is pressed
        break

# Release the camera and close windows
cap.release()
cv2.destroyAllWindows()


# step 2

In [4]:
import cv2
import numpy as np

# Open the laptop's camera
cap = cv2.VideoCapture(0)

while True:
    # Capture frame-by-frame
    ret, frame = cap.read()

    # Convert the frame to grayscale
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Apply Canny edge detection with lower and upper thresholds
    edges = cv2.Canny(gray, 50, 150, apertureSize=3)

    # Probabilistic Hough lines detection
    lines = cv2.HoughLines(edges, 2, np.pi / 180, 200, None, 0, 0)

    if lines is not None:
        # Sort lines based on their distance
        lines = sorted(lines, key=lambda line: line[0][0])

        # Filter out lines that are too close to each other
        filtered_lines = [lines[0]]
        for current_line in lines[1:]:
            distance = abs(current_line[0][0] - filtered_lines[-1][0][0])
            if distance > 10:  # Adjust this threshold
                filtered_lines.append(current_line)

        for line in filtered_lines:
            for rho, theta in line:
                pos_hori = 0
                pos_vert = 0
                a = np.cos(theta)
                b = np.sin(theta)
                x0 = a * rho
                y0 = b * rho
                x1 = int(x0 + 1000 * (-b))
                y1 = int(y0 + 1000 * (a))
                x2 = int(x0 - 1000 * (-b))
                y2 = int(y0 - 1000 * (a))
                # If b > 0.5, the angle must be greater than 45 degrees
                # so we consider that line as a vertical line
                if b > 0.5:
                    # Check the position
                    if rho - pos_hori > 10:
                        # Update the position
                        pos_hori = rho
                        cv2.line(frame, (x1, y1), (x2, y2), (0, 0, 255), 2)
                else:
                    if rho - pos_vert > 10:
                        pos_vert = rho
                        cv2.line(frame, (x1, y1), (x2, y2), (0, 0, 255), 2)

    # Display the original frame with detected lines
    cv2.imshow('Sudoku Table Lines Detection', frame)

    # Check if 's' is pressed
    key = cv2.waitKey(1) & 0xFF
    if key == ord('s'):
        # Save the sudoku lines frame
        cv2.imwrite('sudoku_lines_frame.png', frame)
        
    elif key == ord('q'):
        # Break the loop if 'q' is pressed
        break

# Release the camera and close windows
cap.release()
cv2.destroyAllWindows() 


# step 3 & 4

In [3]:
import cv2

for i in range(1,10): 
    # Load the image
    image = cv2.imread(f'{i}.png')

    # Resize the image to (20, 20)
    resized_image = cv2.resize(image, (50, 50))

    # Save the resized image as pattern_1.png
    cv2.imwrite(f'pattern_{i}.png', resized_image)

In [4]:
import cv2
import numpy as np

def find_pattern_in_image(template_path, image_path):
    # Read the image
    img = cv2.imread(image_path)
    img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Read the template
    template = cv2.imread(template_path)
    height, width = template.shape[:2]

    # Apply Canny edge detection to the template
    edges = cv2.Canny(template, 200, 250)

    # Create the Generalized Hough Transform object
    ght = cv2.createGeneralizedHoughGuil()
    ght.setTemplate(edges)

    # Set other parameters
    ght.setMinDist(100)
    ght.setMinAngle(0)
    ght.setMaxAngle(360)
    ght.setAngleStep(1)
    ght.setLevels(360)
    ght.setMinScale(1)
    ght.setMaxScale(1.3)
    ght.setScaleStep(0.05)
    ght.setAngleThresh(100)
    ght.setScaleThresh(100)
    ght.setPosThresh(100)
    ght.setAngleEpsilon(1)
    ght.setLevels(360)
    ght.setXi(90)

    # Detect the pattern in the image
    positions = ght.detect(img_gray)

    # Check if any positions were found
    if positions and positions[0]:
        # Print the name of the template if a match is found
        print(f"Found pattern {template_path} in the image.")
    else:
        # Print a message if no matches were found
        print(f"No pattern found for {template_path} in the image.")

# List of pattern templates
pattern_templates = ["pattern_1.png", "pattern_2.png", "pattern_3.png",
                     "pattern_4.png", "pattern_5.png", "pattern_6.png",
                     "pattern_7.png", "pattern_8.png", "pattern_9.png"]

# Image to search in
image_to_search = "1.png"

# Iterate over pattern templates and search in the image
for pattern_template in pattern_templates:
    find_pattern_in_image(pattern_template, image_to_search)

No pattern found for pattern_1.png in the image.
No pattern found for pattern_2.png in the image.
No pattern found for pattern_3.png in the image.
No pattern found for pattern_4.png in the image.
No pattern found for pattern_5.png in the image.
No pattern found for pattern_6.png in the image.
No pattern found for pattern_7.png in the image.
No pattern found for pattern_8.png in the image.
No pattern found for pattern_9.png in the image.


In [5]:
import os
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
import cv2

MIN_CANNY_THRESHOLD = 10
MAX_CANNY_THRESHOLD = 50
THRESHOLD = 0.65  # Adjust this parameter based on your requirements

def gradient_orientation(image):
    dx = cv2.Sobel(image, cv2.CV_64F, 1, 0, ksize=3)
    dy = cv2.Sobel(image, cv2.CV_64F, 0, 1, ksize=3)
    gradient = np.arctan2(dy, dx) * 180 / np.pi
    return gradient

def build_r_table(image, origin):
    edges = cv2.Canny(image, MIN_CANNY_THRESHOLD, MAX_CANNY_THRESHOLD)
    gradient = gradient_orientation(edges)

    r_table = defaultdict(list)
    for (i, j), value in np.ndenumerate(edges):
        if value:
            r_table[gradient[i, j]].append((origin[0] - i, origin[1] - j))

    return r_table

def accumulate_gradients(r_table, gray_image):
    edges = cv2.Canny(gray_image, MIN_CANNY_THRESHOLD, MAX_CANNY_THRESHOLD)
    gradient = gradient_orientation(edges)

    accumulator = np.zeros(gray_image.shape)
    for (i, j), value in np.ndenumerate(edges):
        if value:
            for r in r_table[gradient[i, j]]:
                accum_i, accum_j = int(i + r[0]), int(j + r[1])  # Convert to integers
                if 0 <= accum_i < accumulator.shape[0] and 0 <= accum_j < accumulator.shape[1]:
                    accumulator[accum_i, accum_j] += 1

    return accumulator

def post_process_results(accumulator):
    # Thresholding to keep only peaks
    threshold = THRESHOLD * np.max(accumulator)
    binary_result = (accumulator > threshold).astype(np.uint8)

    # Use morphological operations to clean up the result
    kernel = np.ones((5, 5), np.uint8)
    binary_result = cv2.morphologyEx(binary_result, cv2.MORPH_CLOSE, kernel)
    binary_result = cv2.morphologyEx(binary_result, cv2.MORPH_OPEN, kernel)

    return binary_result

def general_hough_closure(reference_image):
    reference_point = (reference_image.shape[0] / 2, reference_image.shape[1] / 2)
    r_table = build_r_table(reference_image, reference_point)

    def f(query_image):
        accumulator = accumulate_gradients(r_table, query_image)
        processed_result = post_process_results(accumulator)
        return processed_result

    return f

def find_pattern_in_image_using_general_hough(template_path, image_path):
    reference_image = cv2.imread(template_path, cv2.IMREAD_GRAYSCALE)
    detect_pattern = general_hough_closure(reference_image)
    query_image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    result = detect_pattern(query_image)

    # Find contours in the result
    contours, _ = cv2.findContours(result, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    # Print the result
    if contours:
        # Use the centroid of the first contour as the result
        M = cv2.moments(contours[0])
        cx = int(M['m10'] / M['m00'])
        cy = int(M['m01'] / M['m00'])
        print(f"Pattern {template_path} found in the image at centroid position ({cx}, {cy}).")
    else:
        print(f"No pattern found for {template_path} in the image.")

# List of pattern templates
pattern_templates = ["pattern_1.png", "pattern_2.png", "pattern_3.png",
                     "pattern_4.png", "pattern_5.png", "pattern_6.png",
                     "pattern_7.png", "pattern_8.png", "pattern_9.png"]

# Image to search in
image_to_search = "1.png"

# Iterate over pattern templates and search in the image
for pattern_template in pattern_templates:
    find_pattern_in_image_using_general_hough(pattern_template, image_to_search)

No pattern found for pattern_1.png in the image.
No pattern found for pattern_2.png in the image.
Pattern pattern_3.png found in the image at centroid position (30, 33).
No pattern found for pattern_4.png in the image.
Pattern pattern_5.png found in the image at centroid position (34, 35).
No pattern found for pattern_6.png in the image.
Pattern pattern_7.png found in the image at centroid position (35, 34).
No pattern found for pattern_8.png in the image.
No pattern found for pattern_9.png in the image.


# online sudoku solver using deep neural network 

### 1. training the deep neural network 


In [41]:
from __future__ import print_function
import keras
from keras.models import Sequential
from keras.layers import Dense, Dropout, Flatten
from keras.layers import Conv2D, MaxPooling2D
from keras.callbacks import LearningRateScheduler
import numpy
import os
import random
import cv2
from scipy import ndimage

# get_best_shift and shift is used to centralize an image according to its center of mass
def get_best_shift(img):
    cy,cx = ndimage.measurements.center_of_mass(img)

    rows,cols = img.shape
    shiftx = numpy.round(cols/2.0-cx).astype(int)
    shifty = numpy.round(rows/2.0-cy).astype(int)

    return shiftx,shifty

def shift(img,sx,sy):
    rows,cols = img.shape
    M = numpy.float32([[1,0,sx],[0,1,sy]])
    shifted = cv2.warpAffine(img,M,(cols,rows))
    return shifted

def shift_according_to_center_of_mass(img):
    img = cv2.bitwise_not(img)
    shiftx,shifty = get_best_shift(img)
    shifted = shift(img,shiftx,shifty)
    img = shifted
    img = cv2.bitwise_not(img)
    return img

batch_size = 128
num_classes = 9
epochs = 25

# input image size 
img_rows, img_cols = 28, 28

DATADIR = "DigitImages"
CATEGORIES = ["1","2","3","4","5","6","7","8","9"]

# Reading training Data 
training_data = []
def create_training_data():
    for category in CATEGORIES:
        path = os.path.join(DATADIR, category)
        class_num = CATEGORIES.index(category)
        for img in os.listdir(path):
            img_array = cv2.imread(os.path.join(path,img), cv2.IMREAD_GRAYSCALE)
            new_array = cv2.resize(img_array, (img_rows, img_cols))
            new_array = shift_according_to_center_of_mass(new_array)
            training_data.append([new_array, class_num])

create_training_data()

random.shuffle(training_data)

# Split 80-20
x_train = []
y_train = []
x_test = []
y_test = []
for i in range(len(training_data)*8//10):
    x_train.append(training_data[i][0])
    y_train.append(training_data[i][1])
for i in range(len(training_data)*8//10,len(training_data)):
    x_test.append(training_data[i][0])
    y_test.append(training_data[i][1])

# Reshape
x_train = numpy.array(x_train)
x_train = x_train.reshape(x_train.shape[0], img_rows, img_cols, 1)
x_test = numpy.array(x_test)
x_test = x_test.reshape(x_test.shape[0], img_rows, img_cols, 1)
input_shape = (img_rows, img_cols, 1)

x_train = x_train.astype('float32')
x_test = x_test.astype('float32')

# Normalizing the data
x_train /= 255
x_test /= 255

# converting class vectors to binary class matrices
y_train = keras.utils.to_categorical(y_train, num_classes)
y_test = keras.utils.to_categorical(y_test, num_classes)

model = Sequential()
model.add(Conv2D(32, kernel_size=(3, 3),
                 activation='relu',
                 input_shape=input_shape))
model.add(Conv2D(64, (3, 3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Flatten())
model.add(Dense(128, activation='relu'))
model.add(Dropout(0.7))
model.add(Dense(num_classes, activation='softmax'))

def lr_schedule(epoch):
    initial_lr = 0.001
    factor = 0.5
    drop_every = 10
    lr = initial_lr * (factor ** (epoch // drop_every))
    return lr

# Creating a learning rate scheduler callback
lr_scheduler = LearningRateScheduler(lr_schedule)

# Compiling the model with the initial learning rate
model.compile(loss=keras.losses.categorical_crossentropy,
              optimizer=keras.optimizers.Adam(0.001),
              metrics=['accuracy'])

# Training the model with the learning rate scheduler callback
model.fit(x_train, y_train,
          batch_size=batch_size,
          epochs=epochs,
          verbose=1,
          validation_data=(x_test, y_test),
          callbacks=[lr_scheduler])

# Evaluating the model
score = model.evaluate(x_test, y_test, verbose=0)
print('Test loss:', score[0])
print('Test accuracy:', score[1])

model.save_weights('digitRecognition.h5')


/tmp/ipykernel_21497/1602985281.py:15: DeprecationWarning: Please use `center_of_mass` from the `scipy.ndimage` namespace, the `scipy.ndimage.measurements` namespace is deprecated.
  cy,cx = ndimage.measurements.center_of_mass(img)


Epoch 1/25
58/58 [==============================] - 5s 73ms/step - loss: 0.9972 - accuracy: 0.6734 - val_loss: 0.1937 - val_accuracy: 0.9426 - lr: 0.0010
Epoch 2/25
58/58 [==============================] - 4s 62ms/step - loss: 0.3410 - accuracy: 0.8953 - val_loss: 0.1084 - val_accuracy: 0.9699 - lr: 0.0010
Epoch 3/25
58/58 [==============================] - 4s 65ms/step - loss: 0.2423 - accuracy: 0.9224 - val_loss: 0.0763 - val_accuracy: 0.9792 - lr: 0.0010
Epoch 4/25
58/58 [==============================] - 4s 65ms/step - loss: 0.1812 - accuracy: 0.9423 - val_loss: 0.0575 - val_accuracy: 0.9841 - lr: 0.0010
Epoch 5/25
58/58 [==============================] - 4s 71ms/step - loss: 0.1494 - accuracy: 0.9506 - val_loss: 0.0465 - val_accuracy: 0.9885 - lr: 0.0010
Epoch 6/25
58/58 [==============================] - 4s 66ms/step - loss: 0.1322 - accuracy: 0.9560 - val_loss: 0.0376 - val_accuracy: 0.9907 - lr: 0.0010
Epoch 7/25
58/58 [==============================] - 4s 68ms/step - loss: 0.1

### 2. defining necessary functions to extract and recognize sudoku cells 

In [1]:
import cv2
import numpy as np
from scipy import ndimage
import math 
import copy

def write_solution_on_image(image, grid, user_grid):
    SIZE = 9
    width = image.shape[1] // 9
    height = image.shape[0] // 9
    for i in range(SIZE):
        for j in range(SIZE):
            if(user_grid[i][j] != 0): 
                continue 
            text = str(grid[i][j])
            off_set_x = width // 15
            off_set_y = height // 15
            font = cv2.FONT_HERSHEY_SIMPLEX
            (text_height, text_width), baseLine = cv2.getTextSize(text, font, fontScale=1, thickness=3)
            font_scale = 0.6 * min(width, height) / max(text_height, text_width)
            text_height *= font_scale
            text_width *= font_scale
            bottom_left_corner_x = width*j + math.floor((width - text_width) / 2) + off_set_x
            bottom_left_corner_y = height*(i+1) - math.floor((height - text_height) / 2) + off_set_y
            image = cv2.putText(image, text, (bottom_left_corner_x, bottom_left_corner_y), 
                                                  font, font_scale, (0,255,0), thickness=3, lineType=cv2.LINE_AA)
    return image

def two_matrices_are_equal(matrix_1, matrix_2, row, col):
    for i in range(row):
        for j in range(col):
            if matrix_1[i][j] != matrix_2[i][j]:
                return False
    return True

def side_lengths_are_too_different(A, B, C, D, eps_scale):
    AB = math.sqrt((A[0]-B[0])**2 + (A[1]-B[1])**2)
    AD = math.sqrt((A[0]-D[0])**2 + (A[1]-D[1])**2)
    BC = math.sqrt((B[0]-C[0])**2 + (B[1]-C[1])**2)
    CD = math.sqrt((C[0]-D[0])**2 + (C[1]-D[1])**2)
    shortest = min(AB, AD, BC, CD)
    longest = max(AB, AD, BC, CD)
    return longest > eps_scale * shortest

def approx_90_degrees(angle, epsilon):
    return abs(angle - 90) < epsilon

def largest_connected_component(image):
    image = image.astype('uint8')
    nb_components, output, stats, centroids = cv2.connectedComponentsWithStats(image, connectivity=8)
    sizes = stats[:, -1]

    if(len(sizes) <= 1):
        blank_image = np.zeros(image.shape)
        blank_image.fill(255)
        return blank_image

    max_label = 1
    max_size = sizes[1]     

    for i in range(2, nb_components):
        if sizes[i] > max_size:
            max_label = i
            max_size = sizes[i]

    img2 = np.zeros(output.shape)
    img2.fill(255)
    img2[output == max_label] = 0

    return img2

def angle_between(vector_1, vector_2):
    unit_vector_1 = vector_1 / np.linalg.norm(vector_1)
    unit_vector2 = vector_2 / np.linalg.norm(vector_2)
    dot_droduct = np.dot(unit_vector_1, unit_vector2)
    angle = np.arccos(dot_droduct)
    return angle * 57.2958 

def get_best_shift(img):
    cy, cx = ndimage.measurements.center_of_mass(img)
    rows, cols = img.shape
    shiftx = np.round(cols/2.0-cx).astype(int)
    shifty = np.round(rows/2.0-cy).astype(int)
    return shiftx, shifty

def shift(img,sx,sy):
    rows,cols = img.shape
    M = np.float32([[1,0,sx],[0,1,sy]])
    shifted = cv2.warpAffine(img,M,(cols,rows))
    return shifted

def get_corners_from_contours(contours, corner_amount=4, max_iter=200):
    coefficient = 1
    while max_iter > 0 and coefficient >= 0:
        max_iter = max_iter - 1
        epsilon = coefficient * cv2.arcLength(contours, True)
        poly_approx = cv2.approxPolyDP(contours, epsilon, True)
        hull = cv2.convexHull(poly_approx)
        if len(hull) == corner_amount:
            return hull
        else:
            if len(hull) > corner_amount:
                coefficient += .01
            else:
                coefficient -= .01
    return None

def prepare(img_array):
    new_array = img_array.reshape(-1, 28, 28, 1)
    new_array = new_array.astype('float32')
    new_array /= 255
    return new_array

def showImage(img, name, width, height):
    new_image = np.copy(img)
    new_image = cv2.resize(new_image, (width, height))
    cv2.imshow(name, new_image)


### 3. extract the cells and solve the sudoku 

In [2]:
import sudokuSolver

def recognize_and_solve_sudoku(image, model, old_sudoku):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5,5), 0)
    thresh = cv2.adaptiveThreshold(blur, 255, 1, 1, 11, 2)

    # Finding contours
    contours, _ = cv2.findContours(thresh, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
    
    # extracting the biggest contour, assuming the Sudoku board is the BIGGEST contour
    max_area = 0
    biggest_contour = None
    for c in contours:
        area = cv2.contourArea(c)
        if area > max_area:
            max_area = area
            biggest_contour = c

    if biggest_contour is None: 
        return image

    corners = get_corners_from_contours(biggest_contour, 4)

    if corners is None: 
        return image

    # locating the top left, top right, bottom left and bottom right corners
    rect = np.zeros((4, 2), dtype = "float32")
    corners = corners.reshape(4,2)

    # Finding top left (sum of coordinates is the smallest)
    sum = 10000
    index = 0
    for i in range(4):
        if(corners[i][0]+corners[i][1] < sum):
            sum = corners[i][0]+corners[i][1]
            index = i
    rect[0] = corners[index]
    corners = np.delete(corners, index, 0)

    # Finding bottom right (sum of coordinates is the biggest)
    sum = 0
    for i in range(3):
        if(corners[i][0]+corners[i][1] > sum):
            sum = corners[i][0]+corners[i][1]
            index = i
    rect[2] = corners[index]
    corners = np.delete(corners, index, 0)

    # Find top right 
    if(corners[0][0] > corners[1][0]):
        rect[1] = corners[0]
        rect[3] = corners[1]
    else:
        rect[1] = corners[1]
        rect[3] = corners[0]

    rect = rect.reshape(4,2)

    # After having found 4 corners A B C D, check if ABCD is approximately square
    #   A------B
    #   |      |
    #   |      |
    #   D------C
    A = rect[0]
    B = rect[1]
    C = rect[2]
    D = rect[3]

    # 1st condition: If all 4 angles are not approximately 90 degrees
    AB = B - A      # 4 vectors AB AD BC DC
    AD = D - A
    BC = C - B
    DC = C - D
    eps_angle = 20
    if not (approx_90_degrees(angle_between(AB,AD), eps_angle) and approx_90_degrees(angle_between(AB,BC), eps_angle)
    and approx_90_degrees(angle_between(BC,DC), eps_angle) and approx_90_degrees(angle_between(AD,DC), eps_angle)):
        return image
    
    # 2nd condition: The Lengths of AB, AD, BC, DC have to be approximately equal
    eps_scale = 1.2 
    if(side_lengths_are_too_different(A, B, C, D, eps_scale)):
        return image

    # the width of the Sudoku board
    (tl, tr, br, bl) = rect
    width_A = np.sqrt(((br[0] - bl[0]) ** 2) + ((br[1] - bl[1]) ** 2))
    width_B = np.sqrt(((tr[0] - tl[0]) ** 2) + ((tr[1] - tl[1]) ** 2))

    # the height of the Sudoku board
    height_A = np.sqrt(((tr[0] - br[0]) ** 2) + ((tr[1] - br[1]) ** 2))
    height_B = np.sqrt(((tl[0] - bl[0]) ** 2) + ((tl[1] - bl[1]) ** 2))

    # taking the maximum of the width and height values to reach the final dimensions
    max_width = max(int(width_A), int(width_B))
    max_height = max(int(height_A), int(height_B))

    # destination points 
    dst = np.array([
	    [0, 0],
	    [max_width - 1, 0],
	    [max_width - 1, max_height - 1],
	    [0, max_height - 1]], dtype = "float32")

    # calculate the perspective transform matrix and warp the perspective to grab the screen
    perspective_transformed_matrix = cv2.getPerspectiveTransform(rect, dst)
    warp = cv2.warpPerspective(image, perspective_transformed_matrix, (max_width, max_height))
    orginal_warp = np.copy(warp)

    # Doing some image processing to get ready for recognizing digits
    warp = cv2.cvtColor(warp,cv2.COLOR_BGR2GRAY) 
    #cv2.imshow("warp", warp)
    warp = cv2.GaussianBlur(warp, (5,5), 0)
    warp = cv2.adaptiveThreshold(warp, 255, 1, 1, 11, 2)
    warp = cv2.bitwise_not(warp)
    _, warp = cv2.threshold(warp, 150, 255, cv2.THRESH_BINARY)

    # Initializing a grid to store Sudoku Board digits
    SIZE = 9
    grid = []
    for i in range(SIZE):
        row = []
        for j in range(SIZE):
            row.append(0)
        grid.append(row)

    height = warp.shape[0] // 9
    width = warp.shape[1] // 9

    offset_width = math.floor(width / 10)    # Offset is used to get rid of the boundaries
    offset_height = math.floor(height / 10)

    # finding the 9*9 matrix of sudoku board cells 
    for i in range(SIZE):
        for j in range(SIZE):

            # Cropping with offset (deleting the boundaries)
            crop_image = warp[height*i+offset_height:height*(i+1)-offset_height, width*j+offset_width:width*(j+1)-offset_width]        
            
            # There are still some boundary lines left. Remove all black lines near the edges
            # If 60% pixels are black, remove
            ratio = 0.6        
            # Top
            while np.sum(crop_image[0]) <= (1-ratio) * crop_image.shape[1] * 255:
                crop_image = crop_image[1:]
            # Bottom
            while np.sum(crop_image[:,-1]) <= (1-ratio) * crop_image.shape[1] * 255:
                crop_image = np.delete(crop_image, -1, 1)
            # Left
            while np.sum(crop_image[:,0]) <= (1-ratio) * crop_image.shape[0] * 255:
                crop_image = np.delete(crop_image, 0, 1)
            # Right
            while np.sum(crop_image[-1]) <= (1-ratio) * crop_image.shape[0] * 255:
                crop_image = crop_image[:-1]    

            crop_image = cv2.bitwise_not(crop_image)
            crop_image = largest_connected_component(crop_image)
           
            digit_pic_size = 28
            crop_image = cv2.resize(crop_image, (digit_pic_size,digit_pic_size))

            # If this is a white cell, set grid[i][j] to 0 and continue on the next image:
            # Criteria 1 for detecting white cell: Has too little black pixels
            if crop_image.sum() >= digit_pic_size**2*255 - digit_pic_size * 1 * 255:
                grid[i][j] == 0
                continue 

            # Criteria 2 for detecting white cell: Huge white area in the center
            center_width = crop_image.shape[1] // 2
            center_height = crop_image.shape[0] // 2
            x_start = center_height // 2
            x_end = center_height // 2 + center_height
            y_start = center_width // 2
            y_end = center_width // 2 + center_width
            center_region = crop_image[x_start:x_end, y_start:y_end]
            
            if center_region.sum() >= center_width * center_height * 255 - 255:
                grid[i][j] = 0
                continue 

            # Applying Binary Threshold to make digits more clear
            _, crop_image = cv2.threshold(crop_image, 200, 255, cv2.THRESH_BINARY) 
            crop_image = crop_image.astype(np.uint8)

            # Centralizing the image according to center of mass
            crop_image = cv2.bitwise_not(crop_image)
            shift_x, shift_y = get_best_shift(crop_image)
            shifted = shift(crop_image,shift_x,shift_y)
            crop_image = shifted
            crop_image = cv2.bitwise_not(crop_image)
            #cv2.imshow(str(i)+str(j), crop_image)

            # Converting to proper format to feed the model 
            crop_image = prepare(crop_image)

            # Recognizing digits
            prediction = model.predict([crop_image]) 
            grid[i][j] = np.argmax(prediction[0]) + 1 

    user_grid = copy.deepcopy(grid)

    # Solving sudoku after recognizing digits of the Sudoku table
    # If this is the same board as last camera frame, print the same solution. No need to solve it again
    if (not old_sudoku is None) and two_matrices_are_equal(old_sudoku, grid, 9, 9):
        if(sudokuSolver.all_board_non_zero(grid)):
            orginal_warp = write_solution_on_image(orginal_warp, old_sudoku, user_grid)
    # If this is a different board, solve it
    else:
        sudokuSolver.solve_sudoku(grid) 
        if(sudokuSolver.all_board_non_zero(grid)): 
            orginal_warp = write_solution_on_image(orginal_warp, grid, user_grid)
            old_sudoku = copy.deepcopy(grid) 

    # Applying inverse perspective transform and printing the solutions on the orginal image
    result_sudoku = cv2.warpPerspective(orginal_warp, perspective_transformed_matrix, (image.shape[1], image.shape[0])
                                        , flags=cv2.WARP_INVERSE_MAP)
    result = np.where(result_sudoku.sum(axis=-1,keepdims=True)!=0, result_sudoku, image)

    return result 


### 4. solve the sudoku online 

In [10]:
import cv2
import numpy as np
from keras.models import Sequential
from keras.layers import Dense, Dropout, Flatten
from keras.layers import Conv2D, MaxPooling2D

def showImage(img, name, width, height):
    new_image = np.copy(img)
    new_image = cv2.resize(new_image, (width, height))
    cv2.imshow(name, new_image)
    out.write(new_image)

# Loading and setting up Camera 
webcam_width, webcam_height = 1920, 1080
cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, webcam_width)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, webcam_height)

# Loading model 
input_shape = (28, 28, 1)
num_classes = 9
model = Sequential()
model.add(Conv2D(32, kernel_size=(3, 3),
                 activation='relu',
                 input_shape=input_shape))
model.add(Conv2D(64, (3, 3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Flatten())
model.add(Dense(128, activation='relu'))
model.add(Dropout(0.7))
model.add(Dense(num_classes, activation='softmax'))

# Loading weights 
model.load_weights("digitRecognition.h5")   

# Defining the codec and create a VideoWriter object
fourcc = cv2.VideoWriter_fourcc(*'XVID')
out = cv2.VideoWriter('sudoku_video.mp4', fourcc, cap.get(cv2.CAP_PROP_FPS), (1600, 900))

# turning on webcam 
old_sudoku = None
while(True):
    ret, frame = cap.read() 
    if ret == True:
        sudoku_frame = recognize_and_solve_sudoku(frame, model, old_sudoku) 
        showImage(sudoku_frame, "Real Time Sudoku Solver", 1600, 900) 

        if cv2.waitKey(1) & 0xFF == ord('q'): 
            break
    else:
        break

cap.release() 
out.release()
cv2.destroyAllWindows() 


OpenCV: FFMPEG: tag 0x44495658/'XVID' is not supported with codec id 12 and format 'mp4 / MP4 (MPEG-4 Part 14)'
OpenCV: FFMPEG: fallback to use tag 0x7634706d/'mp4v'


1/1 [==============================] - 0s 31ms/step


/tmp/ipykernel_8865/3390929681.py:80: DeprecationWarning: Please use `center_of_mass` from the `scipy.ndimage` namespace, the `scipy.ndimage.measurements` namespace is deprecated.
  cy, cx = ndimage.measurements.center_of_mass(img)


1/1 [==============================] - 0s 26ms/step
